# Day 12 – Final Capstone Project



---

## 1. Learning Objectives

- Apply Responsible AI principles: ethics, privacy, bias, and consent.
- Run face detection using OpenCV Haar Cascades on LFW images.
- Build a complete NLP classification pipeline as Capstone Part A.
- Build a complete CNN image classifier as Capstone Part B.
- Save, load, and use a trained model for new predictions (Capstone Part C).
- Design an AI application deployment plan.

## 2. Day Overview

**Total Learning Time: 5 Hours**

| Block | Time | Topic |
|---|---|---|
| Theory | 45 min | Responsible AI, ethics, bias, 12-day recap |
| Lab 1 | 30 min | Face detection with OpenCV Haar Cascades |
| Lab 2 | 30 min | Responsible AI checklist & bias analysis |
| Lab 3 | 45 min | Capstone A: NLP text classification pipeline |
| Lab 4 | 45 min | Capstone B: CNN image classifier |
| Lab 5 | 30 min | Capstone C: Model save, load & deploy |
| Exercises | 20 min | Reflection exercises |
| Review | 15 min | Course summary & Knowledge Check |

## 3. Concept / Theory

### Responsible AI
AI systems can cause real harm if built carelessly. Key principles:

| Principle | Meaning |
|---|---|
| **Fairness** | Model must not discriminate based on protected attributes |
| **Privacy** | Only use consented, anonymized data |
| **Transparency** | Decisions must be explainable |
| **Accountability** | Humans must remain in control |
| **Safety** | Test thoroughly before deployment |

### Face Detection vs Recognition
- **Detection:** Where is a face? (bounding box) — lower risk
- **Recognition:** Who is this person? — HIGH privacy risk, requires strict governance

### 12-Day Curriculum Summary
| Days | Topics |
|---|---|
| 1-2 | Python, NumPy, Pandas |
| 3-4 | ML fundamentals, Scikit-learn |
| 5-6 | Supervised & Unsupervised learning |
| 7-8 | Neural networks, Deep learning (TF/Keras) |
| 9 | Computer vision, CNN, PyTorch |
| 10 | NLP, OCR |
| 11 | Integrated projects |
| **12** | **Capstone + Responsible AI** |

In [ ]:
# Setup
!pip install -q scikit-learn tensorflow opencv-python-headless matplotlib seaborn pandas numpy

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import cv2
import tensorflow as tf
from tensorflow import keras
from sklearn.datasets import fetch_lfw_people
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import warnings; warnings.filterwarnings('ignore')
print('TensorFlow:', tf.__version__)
print('OpenCV:', cv2.__version__)

## 4. Practical Lab 1 – Face Detection with OpenCV Haar Cascades

### Objective
Detect faces in images using OpenCV's Haar Cascade classifier.

### Explanation
Haar Cascades are classical (non-deep-learning) face detectors. They work by scanning an image with a sliding window and applying a series of learned filters. **Important:** We perform detection only — no identity matching or surveillance.

In [ ]:
# Lab 1: Load LFW faces for detection demo
print('Loading LFW dataset...')
lfw = fetch_lfw_people(min_faces_per_person=50, resize=1.0)
print(f'Loaded {lfw.images.shape[0]} face images')

# Display samples
fig, axes = plt.subplots(2, 5, figsize=(14, 6))
for i, ax in enumerate(axes.flat):
    ax.imshow(lfw.images[i*5], cmap='gray')
    ax.set_title(lfw.target_names[lfw.target[i*5]].split()[-1], fontsize=7)
    ax.axis('off')
plt.suptitle('LFW Face Dataset — Detection Demo (no identity matching)')
plt.tight_layout(); plt.show()

In [ ]:
# Lab 1: OpenCV Haar Cascade face detection
# Load the cascade classifier
cascade_path = cv2.data.haarcascades + 'haarcascade_frontalface_default.xml'
detector = cv2.CascadeClassifier(cascade_path)
print('Cascade loaded:', not detector.empty())

def detect_faces(gray_img, scale=1.1, min_neighbors=3, min_size=(20,20)):
    """Detect faces. Returns bounding boxes."""
    faces = detector.detectMultiScale(
        gray_img, scaleFactor=scale,
        minNeighbors=min_neighbors,
        minSize=min_size
    )
    return faces if len(faces) > 0 else []

# Test on a few LFW images
fig, axes = plt.subplots(2, 4, figsize=(14, 7))
for i, ax in enumerate(axes.flat):
    img = lfw.images[i * 10]
    h, w = img.shape
    gray = (img * 255).astype(np.uint8)
    faces = detect_faces(gray)
    vis = cv2.cvtColor(gray, cv2.COLOR_GRAY2RGB)
    for (x, y, fw, fh) in faces:
        cv2.rectangle(vis, (x, y), (x+fw, y+fh), (0, 255, 0), 2)
    ax.imshow(vis)
    ax.set_title(f'Detected: {len(faces)} face(s)', fontsize=8)
    ax.axis('off')
plt.suptitle('Face DETECTION (bounding boxes only — no identity)')
plt.tight_layout(); plt.show()

## 5. Practical Lab 2 – Responsible AI Checklist

### Objective
Evaluate the face detection system through a Responsible AI lens.

### Explanation
Every AI system deployed in the real world must pass a Responsible AI review before launch.

In [ ]:
# Lab 2: Responsible AI Analysis
print('=== RESPONSIBLE AI REVIEW CHECKLIST ===')
print('\n📋 Purpose')
print('  ✓ This system performs face DETECTION only')
print('  ✓ It does NOT identify or verify any person')
print('  ✓ Use case: counting faces in images for research only')

print('\n🔒 Data & Privacy')
print('  ✓ LFW dataset is public-domain research data')
print('  ⚠ All individuals in LFW are public figures who appeared in news media')
print('  ✓ No biometric templates are stored')
print('  ✓ No personal data is logged')

print('\n⚖ Fairness')
print('  ⚠ Haar cascades may perform worse on non-frontal faces')
print('  ⚠ Performance varies by lighting and skin tone')
print('  → Action: Test on diverse demographics before deployment')

print('\n👤 Human Oversight')
print('  ✓ No automated decisions are made from detection results')
print('  ✓ A human reviews all outputs before any action')
print('  ✓ Appeal process: any flagged detection reviewed manually')

print('\n🛑 What This System MUST NOT Do')
print('  ✗ Identify specific individuals')
print('  ✗ Track individuals across time or locations')
print('  ✗ Make decisions affecting people (hiring, access, bail, etc.)')
print('  ✗ Be used for surveillance, policing, or profiling')

In [ ]:
# Bias analysis: detection rate by lighting condition simulation
np.random.seed(42)

# Simulate detection accuracy on different image conditions
conditions = ['Good lighting', 'Low light', 'Side angle', 'Partial occlusion', 'High contrast']
detection_rates = [0.94, 0.71, 0.62, 0.58, 0.83]

plt.figure(figsize=(9, 4))
bars = plt.barh(conditions, detection_rates,
                color=['green' if r >= 0.85 else 'orange' if r >= 0.70 else 'red'
                        for r in detection_rates])
plt.axvline(x=0.85, color='blue', linestyle='--', label='Acceptable threshold (0.85)')
for bar, rate in zip(bars, detection_rates):
    plt.text(bar.get_width() - 0.01, bar.get_y() + bar.get_height()/2,
             f'{rate:.0%}', va='center', ha='right', color='white', fontweight='bold')
plt.xlabel('Detection Rate'); plt.title('Haar Cascade Performance by Condition')
plt.legend(); plt.tight_layout(); plt.show()
print('\n→ Recommendation: DO NOT deploy in low-light or side-angle scenarios')
print('  without additional preprocessing or a more robust detector.')

## 6. Practical Lab 3 – Capstone Part A: NLP Text Classification

### Objective
Build a complete, deployable NLP text classification pipeline.

### Explanation
This integrates Day 10 NLP skills + Day 11 pipeline best practices into a production-ready classifier.

In [ ]:
# Capstone A: Multi-class news classifier (integrated pipeline)
from sklearn.datasets import fetch_20newsgroups
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.metrics import accuracy_score, classification_report, ConfusionMatrixDisplay
import joblib

categories = ['sci.space','sci.med','comp.graphics','rec.sport.hockey','talk.politics.misc']
news = fetch_20newsgroups(subset='all', categories=categories,
                           remove=('headers','footers','quotes'))
print(f'Dataset: {len(news.data)} docs, {len(categories)} categories')

X_tr, X_te, y_tr, y_te = train_test_split(
    news.data, news.target, test_size=0.2, stratify=news.target, random_state=42)

# Build pipeline
nlp_pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(max_features=20000, ngram_range=(1,2),
                               stop_words='english', sublinear_tf=True)),
    ('clf',   LogisticRegression(C=5.0, max_iter=500, random_state=42))
])

# Cross-validate
cv_scores = cross_val_score(nlp_pipeline, X_tr, y_tr, cv=5, scoring='accuracy')
print(f'\n5-Fold CV Accuracy: {cv_scores.mean():.4f} ± {cv_scores.std():.4f}')

# Final train + evaluate
nlp_pipeline.fit(X_tr, y_tr)
y_pred_nlp = nlp_pipeline.predict(X_te)
print(f'Test Accuracy: {accuracy_score(y_te, y_pred_nlp):.4f}')
print('\nClassification Report:')
print(classification_report(y_te, y_pred_nlp, target_names=categories))

In [ ]:
# Confusion matrix & prediction on new text
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

cm = confusion_matrix(y_te, y_pred_nlp)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=[c.split('.')[-1] for c in categories],
            yticklabels=[c.split('.')[-1] for c in categories])
axes[0].set_title('Confusion Matrix — NLP Classifier')
axes[0].set_xlabel('Predicted'); axes[0].set_ylabel('Actual')

# Top words per class
vec = nlp_pipeline.named_steps['tfidf']
clf = nlp_pipeline.named_steps['clf']
feat_names = vec.get_feature_names_out()
top_words = {}
for i, cat in enumerate(categories):
    idx = clf.coef_[i].argsort()[-8:][::-1]
    top_words[cat.split('.')[1]] = [feat_names[j] for j in idx]

axes[1].axis('off')
table_data = [[k, ', '.join(v[:5])] for k, v in top_words.items()]
table = axes[1].table(cellText=table_data, colLabels=['Category', 'Top 5 Words'],
                       loc='center', cellLoc='left')
table.auto_set_font_size(True)
axes[1].set_title('Top Discriminative Words', pad=20)
plt.tight_layout(); plt.show()

# Test with new unseen texts
new_texts = [
    'The NASA telescope discovered a new planet in the solar system',
    'Patients with diabetes need regular blood glucose monitoring',
    'The hockey team won the championship in overtime last night',
]
for text in new_texts:
    pred_idx = nlp_pipeline.predict([text])[0]
    prob = nlp_pipeline.predict_proba([text])[0]
    print(f'{text[:55]}...')
    print(f'  → Predicted: {categories[pred_idx]} (confidence: {prob.max():.3f})')

## 7. Practical Lab 4 – Capstone Part B: CNN Image Classifier

### Objective
Build and train a complete CNN for multi-class image classification.

### Explanation
This integrates Day 8 (TF/Keras) + Day 9 (CNN architecture) skills.

In [ ]:
# Capstone B: CIFAR-10 CNN with augmentation and evaluation
(x_tr, y_tr_c), (x_te, y_te_c) = keras.datasets.cifar10.load_data()
x_tr = x_tr.astype('float32') / 255.0
x_te = x_te.astype('float32') / 255.0
y_tr_c = y_tr_c.squeeze(); y_te_c = y_te_c.squeeze()

cifar_classes = ['airplane','automobile','bird','cat','deer',
                  'dog','frog','horse','ship','truck']

print(f'CIFAR-10: Train={x_tr.shape} | Test={x_te.shape}')
print(f'Classes: {cifar_classes}')

In [ ]:
# Build production-quality CNN
capstone_cnn = keras.Sequential([
    keras.layers.Input(shape=(32, 32, 3)),
    
    keras.layers.Conv2D(32, 3, padding='same', activation='relu'),
    keras.layers.BatchNormalization(),
    keras.layers.Conv2D(32, 3, padding='same', activation='relu'),
    keras.layers.MaxPooling2D(2),
    keras.layers.Dropout(0.25),
    
    keras.layers.Conv2D(64, 3, padding='same', activation='relu'),
    keras.layers.BatchNormalization(),
    keras.layers.Conv2D(64, 3, padding='same', activation='relu'),
    keras.layers.MaxPooling2D(2),
    keras.layers.Dropout(0.35),
    
    keras.layers.Flatten(),
    keras.layers.Dense(256, activation='relu'),
    keras.layers.BatchNormalization(),
    keras.layers.Dropout(0.4),
    keras.layers.Dense(10, activation='softmax')
], name='Capstone_CNN')

capstone_cnn.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)
print('Capstone CNN parameters:', capstone_cnn.count_params())
capstone_cnn.summary()

In [ ]:
# Train (5 epochs for demo; run more for production)
cnn_history = capstone_cnn.fit(
    x_tr, y_tr_c,
    epochs=5,
    batch_size=128,
    validation_split=0.1,
    callbacks=[
        keras.callbacks.EarlyStopping(monitor='val_accuracy', patience=3, restore_best_weights=True),
        keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=2, min_lr=1e-6)
    ],
    verbose=1
)

cnn_loss, cnn_acc = capstone_cnn.evaluate(x_te, y_te_c, verbose=0)
print(f'\nCIFAR-10 Test Accuracy: {cnn_acc:.4f}')

In [ ]:
# Visualize training curves and predictions
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

axes[0].plot(cnn_history.history['accuracy'], label='Train')
axes[0].plot(cnn_history.history['val_accuracy'], label='Validation', ls='--')
axes[0].set_title('Accuracy'); axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].plot(cnn_history.history['loss'], label='Train')
axes[1].plot(cnn_history.history['val_loss'], label='Validation', ls='--')
axes[1].set_title('Loss'); axes[1].legend(); axes[1].grid(alpha=0.3)

# Predictions
y_pred_c = capstone_cnn.predict(x_te[:12], verbose=0).argmax(axis=1)
for i, ax in enumerate(axes[2:2]):
    pass
axes[2].axis('off')
axes[2].text(0.1, 0.5, f'Test Accuracy:\n{cnn_acc:.2%}', fontsize=28, transform=axes[2].transAxes, va='center')
plt.suptitle('Capstone CNN Training Results')
plt.tight_layout(); plt.show()

## 8. Practical Lab 5 – Capstone Part C: Save, Load & Deploy

### Objective
Save trained models, reload them, and build a simple prediction function.

### Explanation
Model deployment means making the trained model available for use in production. The simplest form is saving it to disk and loading it in a prediction script.

In [ ]:
# Capstone C: Save and reload models
import joblib

# Save NLP model
nlp_save_path = 'capstone_nlp_classifier.pkl'
joblib.dump(nlp_pipeline, nlp_save_path)
print(f'NLP model saved: {nlp_save_path}')

# Save CNN model
cnn_save_path = 'capstone_cnn_cifar10.keras'
capstone_cnn.save(cnn_save_path)
print(f'CNN model saved: {cnn_save_path}')

In [ ]:
# Reload and use for inference
loaded_nlp = joblib.load(nlp_save_path)
loaded_cnn  = keras.models.load_model(cnn_save_path)

# NLP inference function
def classify_text(text, model=loaded_nlp, classes=categories):
    pred_idx = model.predict([text])[0]
    prob = model.predict_proba([text])[0]
    return {'category': classes[pred_idx], 'confidence': float(prob.max())}

# CNN inference function
def classify_image(img_array, model=loaded_cnn, classes=cifar_classes):
    """img_array: (32,32,3) float [0,1]"""
    inp = img_array[np.newaxis, ...]
    probs = model.predict(inp, verbose=0)[0]
    top_idx = probs.argmax()
    return {'class': classes[top_idx], 'confidence': float(probs[top_idx])}

# Test NLP
test_texts = [
    'NASA launches new rocket to Mars',
    'Hockey game ended in penalty shootout',
    'New cancer treatment shows promise',
]
print('=== NLP Classifier Predictions ===')
for text in test_texts:
    result = classify_text(text)
    print(f'{text[:50]:50} → {result["category"]:20} ({result["confidence"]:.3f})')

# Test CNN
print('\n=== CNN Image Predictions ===')
for i in range(5):
    result = classify_image(x_te[i])
    true_class = cifar_classes[y_te_c[i]]
    match = '✓' if result['class'] == true_class else '✗'
    print(f'True: {true_class:12} | Pred: {result["class"]:12} | Conf: {result["confidence"]:.3f} {match}')

## 9. Exercises (Reflection)

Answer these questions in Markdown to consolidate your 12-day learning:

**Exercise 1:** What were the 3 most important concepts you learned in this training?

*Your answer here...*

**Exercise 2:** Which day's content was most challenging and why?

*Your answer here...*

**Exercise 3:** Describe a real-world AI application you would build using what you've learned.

*Your answer here...*

**Exercise 4:** What Responsible AI principles would apply to your proposed application?

*Your answer here...*

**Exercise 5:** List 3 skills from this training you want to improve further, and name one resource (library, course, book) for each.

*Your answer here...*

## 10. Challenge Task – AI Application Design

Design a complete AI application for a real-world problem:

1. **Problem statement:** What problem does it solve?
2. **Data:** What data is needed? How will it be collected?
3. **Model:** Which ML/DL approach? (classification, regression, NLP, CV?)
4. **Pipeline:** Describe data flow from input to output
5. **Evaluation:** What metrics will you use?
6. **Responsible AI:** Privacy, fairness, bias mitigation measures
7. **Deployment:** API? Mobile app? Dashboard?
8. **Monitoring:** How will you detect model drift?

Write your design below (no code required — this is an architecture exercise):

In [ ]:
# Challenge Task: Design your AI Application
# Document your design as markdown cells, code cells, or a structured dict

application_design = {
    'name': 'Your Application Name',
    'problem': 'Describe the problem...',
    'data': 'What data will you use...',
    'model_type': 'Classification / Regression / NLP / CV / ...',
    'pipeline': ['Step 1', 'Step 2', 'Step 3', 'Step 4'],
    'metrics': ['Accuracy', 'F1', 'AUC', '...'],
    'responsible_ai': {
        'privacy': 'How you protect privacy...',
        'fairness': 'How you ensure fairness...',
        'transparency': 'How decisions are explained...'
    },
    'deployment': 'API / Mobile / Dashboard / ...',
    'monitoring': 'How you detect model drift...'
}

import json
print(json.dumps(application_design, indent=2))

## 11. Real-World AI/ML Application – What's Next?

### Career Paths in AI/ML

| Role | Focus | Key Skills |
|---|---|---|
| **ML Engineer** | Build & deploy ML systems | Python, MLOps, Cloud, APIs |
| **Data Scientist** | Insights from data | Statistics, EDA, modeling |
| **CV Engineer** | Image/video AI | PyTorch, YOLO, OpenCV |
| **NLP Engineer** | Language AI | Transformers, BERT, LLMs |
| **AI Researcher** | New algorithms | Math, papers, experiments |

### Recommended Next Steps

1. **Deep Learning:** Fast.ai, Deep Learning Specialization (Coursera)
2. **Transformers/LLMs:** HuggingFace Transformers, GPT fine-tuning
3. **MLOps:** MLflow, Docker, cloud (AWS/GCP/Azure)
4. **Computer Vision:** YOLO, Detectron2, Segment Anything
5. **Competitions:** Kaggle, DrivenData
6. **Research:** arXiv, Papers with Code

## 12. Day Summary & Course Completion 🎓

### 12-Day Curriculum Completed!

| Day | Topic | Core Skills Gained |
|---|---|---|
| 1 | Python Fundamentals | Lists, Tuples, Sets, Dicts, Functions |
| 2 | Pandas & NumPy | DataFrames, cleaning, merging |
| 3 | ML Fundamentals | Pipeline, Logistic Regression, metrics |
| 4 | Scikit-learn ML | KNN, Decision Tree, K-Means, Pipeline |
| 5 | Supervised Learning | Classification, Regression, Text ML |
| 6 | Unsupervised Learning | Clustering, PCA, Bandit/RL |
| 7 | Neural Networks | Neurons, activations, backprop |
| 8 | Deep Learning | TF/Keras, Dropout, EarlyStopping, ROC |
| 9 | Computer Vision | CNN, Pillow, PyTorch tensors |
| 10 | NLP & OCR | TF-IDF, EasyOCR, PaddleOCR |
| 11 | Integrated Projects | GridSearch, CV, full pipelines |
| 12 | Capstone | Face detection, Responsible AI, deployment |

**Congratulations on completing the AI/ML Engineering Training Program!**